# Where do product categories and names separate age groups?

Discover the actual breadcrumb content and product-name phrases associated with **Children** and **Infant**, rather than testing depth alone. Compare sibling categories to locate separating branches, then evaluate transparent rules on held-out product groups.

All features are learned from the fitting partition only. Validation is used for reporting, not choosing thresholds or ranking features. These are associations with supplied TargetAgeGroup labels, not medical guidance or verified product suitability. CSVs are read only.


## Results from this run

Held-out minority-age precision and recall (rules chosen on fitting data only):

| Signals | Age | Predicted rows | Precision | Recall |
|---|---|---:|---:|---:|
| category | Children | 84 | 88.1% | 9.8% |
| category | Infant | 7 | 85.7% | 3.7% |
| name | Children | 787 | 92.5% | 96.8% |
| name | Infant | 118 | 98.3% | 72.0% |
| combined | Children | 792 | 91.9% | 96.8% |
| combined | Infant | 116 | 97.4% | 70.2% |

These measurements quantify which labels are recoverable from available category/name content. They do not imply every product has an unambiguous age cue. See the fit-ranked feature tables, sibling splits, and review examples below.


## What actually separates the age labels?

**Product-name age language is the strongest broad signal in this experiment.** In the held-out products, the explicit infant/baby cue matches 141 rows, 138 labeled Infant: **97.9% precision and 85.7% Infant recall**. The children/kids cue matches 777 rows, 710 labeled Children: **91.4% precision and 94.4% Children recall**. These individual cues overlap. They are descriptive validation results, not a newly tuned combined classifier.

**Some breadcrumb branches distinguish Children, but their coverage is narrow.** Walmart's Allergy Medicine branch separates at depth 3 into Childrens Allergy Medicine versus other allergy categories. Its cold/flu branch separates at depth 4 into Childrens Cough Cold and Flu Medicine. CVS has Allergy & Asthma for Children at depth 3. There is no shared universal depth: the age-bearing words and branch matter. Qualified category rules together recover only 9.8% of held-out Children and 3.7% of Infant labels.

**Infant-versus-Children conflicts matter.** The all-rules name system has lower Infant recall (72.0%) than the standalone infant/baby cue because it abstains when qualifying rules disagree. Product titles can mention both kids and babies. Adding category rules does not improve these aggregate holdout results. Do not assume that more matching rules means more confidence.

**A defensible next model:** use product-name age cues as primary evidence, category branches as supporting evidence, and separate conflict/unknown outcomes. Do not default to Adult when no cue is present. Any decision to prioritize an infant cue over a kids cue is a new rule requiring independent validation, not a conclusion established by this holdout.

**Annotation policy remains a question.** Very strong name-label agreement may partly reflect how the original age labels were assigned. Terms such as ?mom baby,? prenatal contexts, and products mentioning both adults and kids need review against the owner's age definitions. These results predict supplied labels; they do not independently validate intended users.


In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
import hashlib, re, math
ROOT=Path.cwd()
if not (ROOT/'data'/'provided').exists(): ROOT=ROOT.parent
DATA=ROOT/'data'/'provided'
raw=pd.read_csv(DATA/'Selfcare_Training_data (1).csv',low_memory=False)
AGES=['Adult','Children','Infant']
MIN_ROWS=10
MIN_GROUPS=8
MIN_PURITY=.80

def path_tokens(value):
    if pd.isna(value): return ()
    t=tuple(re.sub(r'\s+',' ',s).strip().casefold() for s in str(value).split('>') if s.strip())
    for k in range(1,len(t)):
        if len(t)%k==0 and t==t[:k]*(len(t)//k): return t[:k]
    return t

def text_key(value):
    return '' if pd.isna(value) else re.sub(r'\s+',' ',str(value)).strip().casefold()

parent=list(range(len(raw)))
def find(x):
    while parent[x]!=x:
        parent[x]=parent[parent[x]]; x=parent[x]
    return x

def union(a,b):
    a,b=find(a),find(b)
    if a!=b: parent[max(a,b)]=min(a,b)
for col in ['Upc','ProductUrl','ProductName']:
    seen={}
    for idx,value in raw[col].items():
        key=text_key(value)
        if not key: continue
        if key in seen: union(idx,seen[key])
        else: seen[key]=idx

df=raw.loc[raw['TargetAgeGroup'].isin(AGES)].copy()
df['_group']=[find(i) for i in df.index]
df['_retailer']=df['Retailer'].map(text_key)
df['_path']=df['ProductCategory'].map(path_tokens)
is_validation=df['_group'].map(lambda g:int(hashlib.sha256(str(g).encode()).hexdigest()[:8],16)%5==0)
fit=df.loc[~is_validation].copy(); val=df.loc[is_validation].copy()
assert set(fit['_group']).isdisjoint(val['_group'])
print('Fit / validation rows:',len(fit),len(val))
print(pd.DataFrame({'fit':fit['TargetAgeGroup'].value_counts(),'validation':val['TargetAgeGroup'].value_counts()}).reindex(AGES).to_string())
print('Minimum rule support:',MIN_ROWS,'rows and',MIN_GROUPS,'candidate product groups; minimum row AND group purity:',MIN_PURITY)


Fit / validation rows: 62022 15450
                  fit  validation
TargetAgeGroup                   
Adult           58458       14537
Children         2966         752
Infant            598         161
Minimum rule support: 10 rows and 8 candidate product groups; minimum row AND group purity: 0.8


## 1. Define interpretable signals

Breadcrumb features are retailer-specific: individual nodes (at any depth) and complete prefixes. The prefix records the exact branch and depth; the node allows an age-specific category name to appear at multiple depths. Product-name features are pooled across retailers: words and adjacent two-word phrases, excluding pure numbers. Separate fixed regex cues capture common explicit age language and stated years/months for qualitative review.

Connected UPC/URL/exact-name groups reduce duplicate leakage. Incorrect identifiers may over-group; different identifiers may under-group. A group can carry conflicting labels. Rule group purity uses distinct groups per age, so contradictory groups contribute to more than one age count. This is a conservative descriptive measure, not independent trial confidence.


In [2]:
cue_patterns={
 'infant_or_baby':r'\b(?:infants?|infancy|bab(?:y|ies)|newborns?)\b',
 'children_or_kids':r"\b(?:children(?:[?']s)?|child(?:ren)?|kids?|junior|pediatric)\b",
 'toddler':r'\btoddlers?\b',
 'adult':r'\badults?\b',
 'age_in_months':r'\b\d+\s*(?:[-?]\s*\d+\s*)?(?:months?|mos?)\b',
 'age_in_years':r'\b\d+\s*(?:[-?]\s*\d+\s*)?(?:years?|yrs?)\b',
 'prenatal':r'\b(?:prenatal|pregnancy)\b'
}

def features(row):
    retailer=row['_retailer']; path=row['_path']; result=set()
    for depth,node in enumerate(path,1):
        result.add(('node',retailer,node))
        result.add(('prefix',retailer,' > '.join(path[:depth])))
    name=text_key(row['ProductName'])
    words=re.findall(r"[a-z]+(?:['?][a-z]+)?|\d+",name)
    for word in words:
        if len(word)>1 and not word.isdigit(): result.add(('name_word','all',word))
    for a,b in zip(words,words[1:]):
        if not (a.isdigit() and b.isdigit()): result.add(('name_phrase','all',a+' '+b))
    for cue,pattern in cue_patterns.items():
        if re.search(pattern,name): result.add(('name_cue','all',cue))
    return result

feature_sets={i:features(row) for i,row in df.iterrows()}
fit_counts=defaultdict(Counter); fit_groups=defaultdict(lambda:defaultdict(set))
for i,row in fit.iterrows():
    age=row['TargetAgeGroup']; group=row['_group']
    for feature in feature_sets[i]:
        fit_counts[feature][age]+=1
        fit_groups[feature][age].add(group)
val_counts=defaultdict(Counter)
for i,age in val['TargetAgeGroup'].items():
    for feature in feature_sets[i]: val_counts[feature][age]+=1

base=fit['TargetAgeGroup'].value_counts(normalize=True)
fit_age_totals=fit['TargetAgeGroup'].value_counts().to_dict()
validation_age_totals=val['TargetAgeGroup'].value_counts().to_dict()
feature_rows=[]
for feature,counts in fit_counts.items():
    support=sum(counts.values())
    groups=set().union(*fit_groups[feature].values())
    if support<MIN_ROWS or len(groups)<MIN_GROUPS: continue
    group_age_total=sum(len(x) for x in fit_groups[feature].values())
    for age in ['Children','Infant']:
        tp=counts[age]; purity=tp/support
        group_purity=len(fit_groups[feature].get(age,set()))/group_age_total
        observed=val_counts[feature]; v_support=sum(observed.values()); v_tp=observed[age]
        feature_rows.append({'feature':feature,'kind':feature[0],'retailer':feature[1],'signal':feature[2],
         'age':age,'fit_rows':support,'fit_groups':len(groups),'fit_age_rows':tp,
         'fit_precision':purity,'fit_group_precision':group_purity,'fit_lift':purity/base[age],
         'fit_recall':tp/fit_age_totals[age],
         'validation_rows':v_support,'validation_age_rows':v_tp,
         'validation_precision':v_tp/v_support if v_support else np.nan,
         'validation_recall':v_tp/validation_age_totals[age]})
signals=pd.DataFrame(feature_rows)
qualified=signals.loc[signals.fit_precision.ge(MIN_PURITY)&signals.fit_group_precision.ge(MIN_PURITY)].copy()
print('Supported feature/age pairs:',len(signals),'Qualifying pairs:',len(qualified))


Supported feature/age pairs: 40066 Qualifying pairs: 232


## 2. Which breadcrumb content distinguishes each age?

Rank qualifying rules by fitting recall, then fitting precision and support. Validation precision answers ?Of held-out rows matching this signal, how many have this age label?? Validation recall answers ?What fraction of all held-out rows with this age does it recover?? Rules overlap, so individual recall values must not be added. Show weaker enriched signals separately when the 80% requirement leaves gaps.


In [3]:
display_cols=['retailer','signal','fit_rows','fit_groups','fit_precision','fit_group_precision','fit_lift','validation_rows','validation_precision','validation_recall']
for age in ['Children','Infant']:
    print('\nAGE:',age)
    for kind in ['node','prefix']:
        top=qualified.loc[qualified.age.eq(age)&qualified.kind.eq(kind)].sort_values(['fit_recall','fit_precision','fit_rows'],ascending=False).head(12)
        print('\nQualified',kind,'signals:'); print(top[display_cols].to_string(index=False))
    enriched=signals.loc[signals.age.eq(age)&signals.kind.eq('node')&signals.fit_lift.ge(2)].sort_values(['fit_recall','fit_precision'],ascending=False).head(12)
    print('\nEnriched nodes (may be too impure to use as a rule):')
    print(enriched[display_cols].to_string(index=False))



AGE: Children

Qualified node signals:
 retailer                                signal  fit_rows  fit_groups  fit_precision  fit_group_precision  fit_lift  validation_rows  validation_precision  validation_recall
      cvs         cough, cold, flu for children        45          45       0.800000             0.800000 16.728793               13              0.769231           0.013298
  walmart                        kids melatonin        29          27       0.827586             0.814815 17.305648                8              0.875000           0.009309
  walmart childrens cough cold and flu medicine        21          19       0.952381             0.947368 19.915230               12              0.833333           0.013298
  walmart     children's vitamins & supplements        19          19       0.894737             0.894737 18.709834               11              0.909091           0.013298
  walmart                    children's allergy        19          12       0.894737      

## 3. Where do sibling branches separate age?

For every retailer and parent prefix, compare its immediate child branches using fitting data only. Display branches with at least 10 rows and 8 product groups, a sibling with the same minimum support, and at least a 20-percentage-point difference in Children or Infant share. This identifies *where the content changes*, not a universal depth. Mixed labels and label quality still matter.


In [4]:
branches=defaultdict(Counter); branch_groups=defaultdict(set)
for i,row in fit.iterrows():
    path=row['_path']; retailer=row['_retailer']
    for depth in range(1,len(path)):
        key=(retailer,path[:depth],path[depth])
        branches[key][row['TargetAgeGroup']]+=1
        branch_groups[key].add(row['_group'])
parents=defaultdict(list)
for key,count in branches.items():
    if sum(count.values())>=MIN_ROWS and len(branch_groups[key])>=MIN_GROUPS: parents[key[:2]].append(key)
branch_rows=[]
for parent_key,keys in parents.items():
    if len(keys)<2: continue
    for age in ['Children','Infant']:
        shares=[branches[k][age]/sum(branches[k].values()) for k in keys]
        if max(shares)-min(shares)<.20: continue
        for k,share in zip(keys,shares):
            branch_rows.append({'age':age,'retailer':k[0],'parent':' > '.join(k[1]),'child':k[2],
             'child_depth':len(k[1])+1,'rows':sum(branches[k].values()),'groups':len(branch_groups[k]),
             'age_share':share,'sibling_share_range':max(shares)-min(shares)})
branch_table=pd.DataFrame(branch_rows)
if len(branch_table):
    print(branch_table.sort_values(['age','sibling_share_range','retailer','parent','age_share'],ascending=[True,False,True,True,False]).to_string(index=False))
else: print('No sibling splits met the predeclared thresholds.')


     age  retailer                                                                                                   parent                                                 child  child_depth  rows  groups  age_share  sibling_share_range
Children   walmart                                                                   health and medicine > allergy medicine                            childrens allergy medicine            3    12      11   1.000000             1.000000
Children   walmart                                                                   health and medicine > allergy medicine                                              claritin            3    11      11   0.272727             1.000000
Children   walmart                                                                   health and medicine > allergy medicine                             best medicine for allergy            3    69      61   0.130435             1.000000
Children   walmart                                  

## 4. Do product-name cues resolve age when the category does not?

Learn words/phrases from the fitting partition using the same support and purity criteria. These can include brands or product forms and should not automatically be interpreted as causal age attributes. The fixed explicit-cue table also shows label mixtures and counterexamples; ?baby,? ?prenatal,? or ?children? in a name does not guarantee the supplied age label.


In [5]:
for age in ['Children','Infant']:
    top=qualified.loc[qualified.age.eq(age)&qualified.kind.str.startswith('name')].sort_values(['fit_recall','fit_precision','fit_rows'],ascending=False).head(20)
    print('\nName signals for',age); print(top[['kind']+display_cols].to_string(index=False))
cue_rows=[]
for cue in cue_patterns:
    key=('name_cue','all',cue)
    for split,counts in [('fit',fit_counts[key]),('validation',val_counts[key])]:
        cue_rows.append({'cue':cue,'split':split,'rows':sum(counts.values()),**{age:counts[age] for age in AGES}})
print('\nExplicit age cues and supplied labels:')
print(pd.DataFrame(cue_rows).to_string(index=False))



Name signals for Children
       kind retailer             signal  fit_rows  fit_groups  fit_precision  fit_group_precision  fit_lift  validation_rows  validation_precision  validation_recall
   name_cue      all   children_or_kids      3052        2877       0.915793             0.912183 19.150138              777              0.913771           0.944149
  name_word      all               kids      2244        2128       0.939840             0.937089 19.652977              544              0.939338           0.679521
name_phrase      all           for kids       856         848       0.925234             0.924528 19.347553              218              0.922018           0.267287
  name_word      all         children's       443         402       1.000000             1.000000 20.910991              127              1.000000           0.168883
  name_word      all           children       429         421       0.951049             0.950119 19.887376              114              0.938

## 5. Evaluate category-only, name-only, and combined rules

Each system uses all qualifying fitting rules from its feature family. Predict a minority age only if matching rules agree; abstain on conflicting Children/Infant rules or no qualifying signal. Adult is not predicted: this is targeted recovery of minority labels, with an explicit unknown outcome. Report per-age precision, recall, and true-positive candidate product-group counts, rather than majority-dominated overall accuracy.

The combined system also reports how many correct held-out cases name cues add where category rules abstain. Signals are not tuned on this holdout. Results describe these supplied labels and this split, not confirmed intended product use.


In [6]:
rule_map={row.feature:row.age for row in qualified.itertuples()}
def predict(index,family):
    matched=[(f,rule_map[f]) for f in feature_sets[index] if f in rule_map and
             (family=='combined' or (family=='category' and f[0] in ['node','prefix']) or
              (family=='name' and f[0].startswith('name')))]
    ages={age for _,age in matched}
    if len(ages)==1: return next(iter(ages)), ' | '.join(sorted(f[0]+': '+f[2] for f,_ in matched))
    return ('Conflict' if ages else 'Abstain'), ''
predictions={}; results=[]
for family in ['category','name','combined']:
    output=pd.DataFrame([predict(i,family) for i in val.index],index=val.index,columns=['prediction','evidence'])
    predictions[family]=output
    print('\n'+family+' confusion table:')
    print(pd.crosstab(val['TargetAgeGroup'],output['prediction']).reindex(index=AGES,fill_value=0).to_string())
    for age in ['Children','Infant']:
        predicted=output.prediction.eq(age); actual=val.TargetAgeGroup.eq(age); correct=predicted&actual
        results.append({'system':family,'age':age,'predicted_rows':int(predicted.sum()),'true_positive_rows':int(correct.sum()),
         'true_positive_groups':val.loc[correct,'_group'].nunique(),'precision':correct.sum()/predicted.sum() if predicted.sum() else np.nan,
         'recall':correct.sum()/actual.sum(),'false_positive_rows':int((predicted&~actual).sum())})
metrics=pd.DataFrame(results)
print('\nHeld-out performance:'); print(metrics.to_string(index=False))
for age in ['Children','Infant']:
    added=val.TargetAgeGroup.eq(age)&predictions['category'].prediction.eq('Abstain')&predictions['combined'].prediction.eq(age)
    print('Additional correct',age,'rows from combined rules where category abstained:',int(added.sum()))



category confusion table:
prediction      Abstain  Children  Infant
TargetAgeGroup                           
Adult             14530         6       1
Children            678        74       0
Infant              151         4       6

name confusion table:
prediction      Abstain  Children  Conflict  Infant
TargetAgeGroup                                     
Adult             14478        57         0       2
Children             16       728         8       0
Infant                5         2        38     116

combined confusion table:
prediction      Abstain  Children  Conflict  Infant
TargetAgeGroup                                     
Adult             14472        62         0       3
Children             16       728         8       0
Infant                4         2        42     113

Held-out performance:
  system      age  predicted_rows  true_positive_rows  true_positive_groups  precision   recall  false_positive_rows
category Children              84                  74

## 6. Inspect useful evidence, false positives, and unresolved products

Keep source-row indices. Incorrect predictions may reflect an unreliable cue, a product-context ambiguity, or a questionable supplied age annotation; these tables do not decide which. Review residual cases before proposing more rules.


In [7]:
combined=predictions['combined']
for age in ['Children','Infant']:
    true_age=val.TargetAgeGroup.eq(age); predicted=combined.prediction.eq(age)
    selections={'correct':true_age&predicted,'false_positive':~true_age&predicted,
                'unresolved_or_wrong':true_age&~predicted}
    for label,mask in selections.items():
        print('\n'+age+': '+label)
        print(val.loc[mask,['Retailer','ProductName','ProductCategory','TargetAgeGroup']].join(combined).head(12).to_string(max_colwidth=135))



Children: correct
       Retailer                                                                                                                             ProductName                                                                                                 ProductCategory TargetAgeGroup prediction                                                                                                                                evidence
190      Amazon  BebirdPro Ear Wax Removal Cleaner for Kids, Ear Cleaner with HD Camera, Ear Check Camera Otoscope Wax Remover for iOS, Android Phon...                                                    Health & Household > Health Care > Ear Care > Earwax Removal       Children   Children                                            name_cue: children_or_kids | name_phrase: for kids | name_phrase: kids ear | name_word: kids
197   Walgreens                                                                                                       Dr. Formula

## Limits and next step

Use age-bearing category nodes and product-name phrases as evidence features, with retailer context. They can provide focused age predictions when supported, and abstention elsewhere. Absence of a cue does not imply Adult. Depth is only the location of a separating branch, not an age rule itself.

The one holdout is not a tuning set. Before production use, audit discordant labels, confirm age definitions (especially Infant versus Children), and evaluate on another independent product-group partition. Candidate identity grouping and annotation quality limit certainty. Multiple overlapping signals are correlated and their precision cannot be multiplied.
